In [12]:
import pandas as pd
import json
import os
import numpy as np
from datasets import Dataset

from transformers import (
    set_seed,
)

from time import time
import pickle
import matplotlib.pyplot as plt
import random
from tqdm import tqdm
from unsloth import FastLanguageModel
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import torch
from datasets import load_dataset
def_seed = 42

set_seed(def_seed)
np.random.seed(def_seed)
import random
random.seed(def_seed)


device = 'cuda' if torch.cuda.is_available() else 'cpu'


Selecting batch size

In [4]:
batch_size = 50

Zero-shot prompt tempalte for 2 label task: Hate + Neutral

In [2]:
prompt_template = '''You are tasked with annotating speech. Your response must be a single valid number:
1 for Hate/Offensive/Sexism/Toxic/Political/COVID-related Hate Speech,
2 for Normal Speech.

Provide only the number corresponding to the category. Do not include any explanation or additional text.
Do you think the following comment is Hate/Offensive/Sexism/Toxic/Political/COVID-related Hate Speech, speech or Normal speech?
\n"{comment}"\n
Your Answer:
'''

prompt_template = '''You are tasked with annotating speech. Your response must be a single valid number:
1 for Hate/Offensive/Sexism/Toxic/Political/COVID-related Hate Speech,
2 for Normal Speech.

Provide only the number corresponding to the category. Do not include any explanation or additional text.
Do you think the following comment is Hate/Offensive/Sexism/Toxic/Political/COVID-related Hate Speech, speech or Normal speech?
\n"{comment}"\n
Your Answer:
'''

A list of fine-tuned models is available on Hugging Face

In [ ]:
models_list_HF = {"Llama3.2-1B": {
                    "Base": "unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit",
                    "Human": "anonymousOWSHateLLM/Hate-Llama3.2-1B.human.2_label",
                    "Lgb": "anonymousOWSHateLLM/Hate-Llama3.2-1B.Lgb.2_label",
                    "Mean": "anonymousOWSHateLLM/Hate-Llama3.2-1B.Mean.2_label",
                    "Vote": "anonymousOWSHateLLM/Hate-Llama3.2-1B.Vote.2_label",
                    "Human-Lgb": "anonymousOWSHateLLM/Hate-Llama3.2-1B.Human_Lgb.2_label",
                    },
                "Qwen2.5-14B": {
                    "Base": "unsloth/Qwen2.5-14B-Instruct-bnb-4bit",
                    "Human": "anonymousOWSHateLLM/Hate-Qwen2.5-14B.Human.2_label",
                    "Lgb": "anonymousOWSHateLLM/Hate-Qwen2.5-14B.Lgb.2_label",
                    "Mean": "anonymousOWSHateLLM/Hate-Qwen2.5-14B.Mean.2_label",
                    "Vote": "anonymousOWSHateLLM/Hate-Qwen2.5-14B.Vote.2_label",
                    "Human-Lgb": "anonymousOWSHateLLM/Hate-Qwen2.5-14B.Human_Lgb.2_label",
                    }}


Evaluation set: Two group 1 and group 2

Load the report and df_eval_set_1 and df_eval_set_1 locally with the probs

In [14]:
import pickle
with open('report_all.pkl', 'rb') as f:
    report_all = pickle.load(f)
report_all.keys()

dict_keys(['Qwen2.5-14B_test_1', 'Qwen2.5-14B_test_2', 'Llama3.2-1B_test_1', 'Llama3.2-1B_test_2'])

In [15]:
df_eval_set_1  = pd.read_csv('df_eval_set_1.csv')
df_eval_set_2  = pd.read_csv('df_eval_set_2.csv')
df_eval_set_1.loc[0]

text                                   Hey Twitter, since y’all like to get so involv...
ds                                                                                 Covid
language                                                                             eng
label_id                                                                               2
Qwen2.5-14B_Base_probs_label_1                                                       1.0
Qwen2.5-14B_Base_probs_label_2                                                       0.0
Qwen2.5-14B_Human_probs_label_1                                                  0.18262
Qwen2.5-14B_Human_probs_label_2                                                  0.81641
Qwen2.5-14B_Lgb_probs_label_1                                                    0.02039
Qwen2.5-14B_Lgb_probs_label_2                                                    0.98047
Qwen2.5-14B_Mean_probs_label_1                                                   0.00406
Qwen2.5-14B_Mean_prob

In [16]:
df_eval_set_2.loc[0]

text                                   RT @Fucking_Garza: I wanna sleep but I'm outsi...
ds                                                                                SetFit
language                                                                             eng
label_id                                                                               1
Qwen2.5-14B_Base_probs_label_1                                                       1.0
Qwen2.5-14B_Base_probs_label_2                                                       0.0
Qwen2.5-14B_Human_probs_label_1                                                  0.96094
Qwen2.5-14B_Human_probs_label_2                                                  0.03735
Qwen2.5-14B_Lgb_probs_label_1                                                    0.95703
Qwen2.5-14B_Lgb_probs_label_2                                                    0.04199
Qwen2.5-14B_Mean_probs_label_1                                                       1.0
Qwen2.5-14B_Mean_prob

In [17]:
dataset = load_dataset("anonymousOWSHateLLM/Hate.2_label_eval_data")

README.md:   0%|          | 0.00/489 [00:00<?, ?B/s]

group_1-00000-of-00001.parquet:   0%|          | 0.00/457k [00:00<?, ?B/s]

group_2-00000-of-00001.parquet:   0%|          | 0.00/499k [00:00<?, ?B/s]

Generating group_1 split:   0%|          | 0/5481 [00:00<?, ? examples/s]

Generating group_2 split:   0%|          | 0/5700 [00:00<?, ? examples/s]

In [18]:
dataset

DatasetDict({
    group_1: Dataset({
        features: ['text', 'ds', 'language', 'label_id'],
        num_rows: 5481
    })
    group_2: Dataset({
        features: ['text', 'ds', 'language', 'label_id'],
        num_rows: 5700
    })
})

In [19]:
df_eval_set_1 = pd.DataFrame(dataset["group_1"])
print("Total Samples of set 1: ",df_eval_set_1.shape[0])
print(df_eval_set_1.groupby(['ds', 'label_id']).size().unstack(fill_value=0))

Total Samples of set 1:  5481
label_id        1    2
ds                    
Covid          76  317
GermEval2019  343  657
GermEval2021  218  446
HateSpeechX   612  388
Sexism        152  848
US_election    48  376
ViHSD         176  824


In [20]:
df_eval_set_2 = pd.DataFrame(dataset["group_2"])
print("Total Samples of set 2: ",df_eval_set_2.shape[0])
print(df_eval_set_2.groupby(['ds', 'label_id']).size().unstack(fill_value=0))

Total Samples of set 2:  5700
label_id                1    2
ds                            
Ethos                 428  556
GermanEval18          380  497
Hasoc                 440  436
HateOff               500  500
Manueltonneau_German  491  472
SetFit                500  500


Selecting Group model Llama3.2-1B or Qwen2.5-14B 
Test set: 1 or 2

In [10]:
model_group = "Llama3.2-1B"


In [9]:
def process_task(texts, model, tokenizer, stop_token_id):
    encoding = tokenizer(texts, padding=True, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model(**encoding)
        logits = outputs.logits  # Shape: [batch_size, sequence_length, vocab_size]
    last_token_logits = logits[:, -1, :]  # Shape: [vocab_size]
    probabilities = torch.softmax(last_token_logits, dim=-1)
    indices = torch.tensor(stop_token_id)
    selected_probs_1 = probabilities[:, indices[0]].float().cpu().numpy()
    selected_probs_2 = probabilities[:, indices[1]].float().cpu().numpy()
    return selected_probs_1, selected_probs_2

In [8]:
def preprocess(text, model_id, tokenizer):
    user_message_content = prompt_template.format(comment=text)
    user_message = {
        "role": "user",
        "content": user_message_content
    }

    if "Qwen" in model_id:
        system_message =  {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant"}
    else:
        system_message =  {"role": "system", "content": "You are a helpful assistant"}
    messages = [system_message, user_message]
    messages = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    messages = messages


    return messages


In [7]:
def run_eval(group_test=1):
    model_probs_dict = {}

    model_list = models_list_HF[model_group]
    df_eval = df_eval_set_1 if group_test == 1 else df_eval_set_2


    for key, model_id in model_list.items():

        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_id,
            max_seq_length=500,
            dtype=getattr(torch, "bfloat16"),
        )
        FastLanguageModel.for_inference(model)
        tokenizer.padding_side = "left"

        stop_token_id = [16, 17]

        if model_group == "Qwen2.5-14B":
            stop_token_id = tokenizer(["12"])['input_ids'][0]
        elif model_group == "Llama3.2-1B":
            stop_token_id = [16, 17]


        df_eval["prompt"] = df_eval["text"].apply(lambda text: preprocess(text, model_id, tokenizer))


        texts = []
        probs_label_1 = []
        probs_label_2 = []

        prompts = df_eval['prompt'].tolist()

        for i in tqdm(range(0, len(prompts), batch_size)):
            batch = prompts[i:i+batch_size]
            selected_probs_1, selected_probs_2 = process_task(batch, model, tokenizer, stop_token_id)
            probs_label_1.extend(selected_probs_1.tolist())
            probs_label_2.extend(selected_probs_2.tolist())
            torch.cuda.empty_cache()
            torch.cuda.synchronize()


        model_probs_dict[key] = {
            "probs_label_1": probs_label_1,
            "probs_label_2": probs_label_2,
        }

    report = {}
    for ds in df_eval['ds'].unique():
        report[ds] = {}
    report["Mean"] = {}

    y_true = np.array(df_eval['label_id'] == 1, dtype=int)
    df_eval['y_ture'] = y_true

    for model, probs in model_probs_dict.items():
        y_prob_label_1 = probs["probs_label_1"]

        best_f1 = 0
        best_threshold = 0
        for threshold in np.arange(0.01, 0.99, 0.02):
            f1_s = f1_score(y_true, y_prob_label_1 > threshold)
            if f1_s > best_f1:
                best_f1, best_threshold = f1_s, threshold
        df_eval['y_pred'] = y_prob_label_1 > best_threshold
        df_eval['probs'] = y_prob_label_1

        report["Mean"][model] = {
        "auc": round(roc_auc_score(y_true, y_prob_label_1), 3), 
        "acc": round(accuracy_score(y_true, df_eval['y_pred'])* 100, 1), 
        "f1": round(f1_score(y_true,df_eval['y_pred'])* 100, 1)
            }
        
        for ds in df_eval['ds'].unique():
            tmp_df = df_eval.loc[df_eval['ds'] == ds]

            best_f1 = 0
            best_threshold = 0

            for threshold in np.arange(0.01, 0.99, 0.02):
                f1_s = f1_score(tmp_df['y_ture'], tmp_df['probs'] > threshold)
                if f1_s > best_f1:
                    best_f1, best_threshold = f1_s, threshold

            y_pred = tmp_df['probs'] >= best_threshold


            auc = round(roc_auc_score(tmp_df['y_ture'], tmp_df['probs']), 3)
            acc = round(accuracy_score(tmp_df['y_ture'],y_pred)* 100, 1) 
            f1 = round(f1_score(tmp_df['y_ture'], y_pred)* 100, 1) 
            report[ds][model] = {
                            "auc": auc, 
                            "acc": acc, 
                            "f1": f1
                            }
    
    report_df = pd.DataFrame(report).T
    return model_probs_dict, report_df

In [ ]:
model_group = "Llama3.2-1B"
result = {}
for group_test in [1, 2]:
    result[group_test] = {}
    probs, report = run_eval(group_test)
    result[group_test]["probs"] = probs
    result[group_test]["report"] = report

report_all = {}

report_all["Llama3.2-1B_test_1"] = result[1]["report"]
report_all["Llama3.2-1B_test_2"] = result[2]["report"]

==((====))==  Unsloth 2025.2.5: Fast Llama patching. Transformers: 4.48.3.
   \\   /|    GPU: NVIDIA A100 80GB PCIe. Max memory: 79.252 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29. FA2 = True]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


100%|█████████████████████████████████████████████████████████████| 110/110 [00:23<00:00,  4.66it/s]


==((====))==  Unsloth 2025.2.5: Fast Qwen2 patching. Transformers: 4.48.3.
   \\   /|    GPU: NVIDIA A100 80GB PCIe. Max memory: 79.252 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29. FA2 = True]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

100%|█████████████████████████████████████████████████████████████| 110/110 [03:03<00:00,  1.67s/it]


...

100%|█████████████████████████████████████████████████████████████| 114/114 [00:29<00:00,  3.90it/s]


==((====))==  Unsloth 2025.2.5: Fast Llama patching. Transformers: 4.48.3.
   \\   /|    GPU: NVIDIA A100 80GB PCIe. Max memory: 79.252 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29. FA2 = True]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


100%|█████████████████████████████████████████████████████████████| 114/114 [00:29<00:00,  3.93it/s]


==((====))==  Unsloth 2025.2.5: Fast Llama patching. Transformers: 4.48.3.
   \\   /|    GPU: NVIDIA A100 80GB PCIe. Max memory: 79.252 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29. FA2 = True]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


100%|█████████████████████████████████████████████████████████████| 114/114 [00:28<00:00,  3.97it/s]


In [16]:
result[1]["report"]

,Base,Human,Lgb,Mean,Vote,Human-Lgb
Covid,"{'auc': 0.53, 'acc': 26.5, 'f1': 33.3}","{'auc': 0.79, 'acc': 74.8, 'f1': 51.7}","{'auc': 0.792, 'acc': 77.6, 'f1': 52.7}","{'auc': 0.808, 'acc': 80.4, 'f1': 52.8}","{'auc': 0.81, 'acc': 78.1, 'f1': 53.3}","{'auc': 0.847, 'acc': 76.6, 'f1': 57.8}"
GermEval2019,"{'auc': 0.582, 'acc': 44.6, 'f1': 52.3}","{'auc': 0.649, 'acc': 44.9, 'f1': 53.6}","{'auc': 0.759, 'acc': 70.5, 'f1': 62.9}","{'auc': 0.65, 'acc': 54.1, 'f1': 56.2}","{'auc': 0.655, 'acc': 53.5, 'f1': 57.1}","{'auc': 0.786, 'acc': 69.3, 'f1': 63.7}"
GermEval2021,"{'auc': 0.586, 'acc': 43.2, 'f1': 51.7}","{'auc': 0.542, 'acc': 37.8, 'f1': 50.4}","{'auc': 0.658, 'acc': 49.2, 'f1': 53.0}","{'auc': 0.618, 'acc': 47.6, 'f1': 52.2}","{'auc': 0.627, 'acc': 45.8, 'f1': 52.0}","{'auc': 0.639, 'acc': 49.1, 'f1': 52.0}"
HateSpeechX,"{'auc': 0.563, 'acc': 61.4, 'f1': 76.0}","{'auc': 0.667, 'acc': 62.1, 'f1': 76.2}","{'auc': 0.719, 'acc': 68.4, 'f1': 78.4}","{'auc': 0.713, 'acc': 67.8, 'f1': 77.8}","{'auc': 0.72, 'acc': 70.3, 'f1': 78.5}","{'auc': 0.758, 'acc': 68.0, 'f1': 78.3}"
Sexism,"{'auc': 0.46, 'acc': 18.4, 'f1': 26.6}","{'auc': 0.703, 'acc': 61.4, 'f1': 36.7}","{'auc': 0.695, 'acc': 59.7, 'f1': 36.1}","{'auc': 0.655, 'acc': 53.0, 'f1': 32.3}","{'auc': 0.625, 'acc': 55.5, 'f1': 31.2}","{'auc': 0.815, 'acc': 72.5, 'f1': 47.0}"
US_election,"{'auc': 0.604, 'acc': 65.6, 'f1': 24.7}","{'auc': 0.68, 'acc': 73.3, 'f1': 29.8}","{'auc': 0.77, 'acc': 85.1, 'f1': 37.6}","{'auc': 0.789, 'acc': 79.7, 'f1': 40.3}","{'auc': 0.796, 'acc': 85.1, 'f1': 43.2}","{'auc': 0.78, 'acc': 79.5, 'f1': 41.6}"
ViHSD,"{'auc': 0.629, 'acc': 55.6, 'f1': 34.1}","{'auc': 0.655, 'acc': 46.7, 'f1': 34.9}","{'auc': 0.736, 'acc': 80.0, 'f1': 45.1}","{'auc': 0.721, 'acc': 71.0, 'f1': 43.1}","{'auc': 0.725, 'acc': 73.0, 'f1': 42.8}","{'auc': 0.748, 'acc': 68.2, 'f1': 43.6}"
Mean,"{'auc': 0.527, 'acc': 29.7, 'f1': 45.8}","{'auc': 0.711, 'acc': 60.9, 'f1': 53.5}","{'auc': 0.768, 'acc': 72.4, 'f1': 58.8}","{'auc': 0.722, 'acc': 65.5, 'f1': 55.5}","{'auc': 0.716, 'acc': 62.7, 'f1': 55.2}","{'auc': 0.798, 'acc': 72.8, 'f1': 60.8}"


In [17]:
result[2]["report"]

,Base,Human,Lgb,Mean,Vote,Human-Lgb
SetFit,"{'auc': 0.546, 'acc': 50.0, 'f1': 66.7}","{'auc': 0.773, 'acc': 65.9, 'f1': 72.7}","{'auc': 0.894, 'acc': 81.8, 'f1': 83.7}","{'auc': 0.891, 'acc': 81.7, 'f1': 83.2}","{'auc': 0.888, 'acc': 82.0, 'f1': 83.6}","{'auc': 0.881, 'acc': 79.3, 'f1': 80.9}"
Manueltonneau_German,"{'auc': 0.479, 'acc': 51.1, 'f1': 67.6}","{'auc': 0.694, 'acc': 60.7, 'f1': 69.8}","{'auc': 0.58, 'acc': 54.9, 'f1': 68.7}","{'auc': 0.619, 'acc': 56.2, 'f1': 68.2}","{'auc': 0.621, 'acc': 57.6, 'f1': 68.5}","{'auc': 0.658, 'acc': 60.7, 'f1': 69.3}"
HateOff,"{'auc': 0.581, 'acc': 49.9, 'f1': 66.6}","{'auc': 0.759, 'acc': 65.5, 'f1': 72.7}","{'auc': 0.883, 'acc': 81.2, 'f1': 82.9}","{'auc': 0.872, 'acc': 79.7, 'f1': 81.2}","{'auc': 0.863, 'acc': 78.7, 'f1': 80.7}","{'auc': 0.871, 'acc': 78.9, 'f1': 80.1}"
Ethos,"{'auc': 0.605, 'acc': 44.7, 'f1': 60.9}","{'auc': 0.751, 'acc': 62.6, 'f1': 67.8}","{'auc': 0.782, 'acc': 71.4, 'f1': 72.3}","{'auc': 0.795, 'acc': 69.3, 'f1': 71.8}","{'auc': 0.792, 'acc': 68.6, 'f1': 71.6}","{'auc': 0.823, 'acc': 72.9, 'f1': 74.1}"
Hasoc,"{'auc': 0.631, 'acc': 53.4, 'f1': 67.8}","{'auc': 0.621, 'acc': 50.7, 'f1': 67.0}","{'auc': 0.699, 'acc': 61.3, 'f1': 69.9}","{'auc': 0.671, 'acc': 57.1, 'f1': 68.8}","{'auc': 0.673, 'acc': 57.2, 'f1': 68.9}","{'auc': 0.676, 'acc': 59.5, 'f1': 70.2}"
GermanEval18,"{'auc': 0.55, 'acc': 43.4, 'f1': 60.5}","{'auc': 0.549, 'acc': 43.3, 'f1': 60.5}","{'auc': 0.626, 'acc': 51.9, 'f1': 62.0}","{'auc': 0.618, 'acc': 52.5, 'f1': 62.9}","{'auc': 0.612, 'acc': 53.4, 'f1': 62.2}","{'auc': 0.648, 'acc': 57.9, 'f1': 61.0}"
Mean,"{'auc': 0.555, 'acc': 48.0, 'f1': 64.9}","{'auc': 0.661, 'acc': 56.1, 'f1': 67.7}","{'auc': 0.706, 'acc': 65.2, 'f1': 72.3}","{'auc': 0.678, 'acc': 63.3, 'f1': 71.3}","{'auc': 0.674, 'acc': 64.0, 'f1': 71.4}","{'auc': 0.735, 'acc': 65.9, 'f1': 72.0}"


Qwen2.5-14B

In [19]:
model_group = "Qwen2.5-14B"

result = {}
for group_test in [1, 2]:
    result[group_test] = {}
    probs, report = run_eval(group_test)
    result[group_test]["probs"] = probs
    result[group_test]["report"] = report


==((====))==  Unsloth 2025.2.5: Fast Qwen2 patching. Transformers: 4.48.3.
   \\   /|    GPU: NVIDIA A100 80GB PCIe. Max memory: 79.252 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29. FA2 = True]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

100%|█████████████████████████████████████████████████████████████| 110/110 [02:46<00:00,  1.52s/it]


==((====))==  Unsloth 2025.2.5: Fast Qwen2 patching. Transformers: 4.48.3.
   \\   /|    GPU: NVIDIA A100 80GB PCIe. Max memory: 79.252 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29. FA2 = True]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

...

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

100%|█████████████████████████████████████████████████████████████| 114/114 [03:39<00:00,  1.92s/it]


==((====))==  Unsloth 2025.2.5: Fast Qwen2 patching. Transformers: 4.48.3.
   \\   /|    GPU: NVIDIA A100 80GB PCIe. Max memory: 79.252 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29. FA2 = True]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

100%|█████████████████████████████████████████████████████████████| 114/114 [03:39<00:00,  1.93s/it]


In [20]:
result[1]["report"]

,Base,Human,Lgb,Mean,Vote,Human-Lgb
Covid,"{'auc': 0.64, 'acc': 41.0, 'f1': 39.6}","{'auc': 0.865, 'acc': 86.5, 'f1': 61.3}","{'auc': 0.887, 'acc': 85.0, 'f1': 64.2}","{'auc': 0.89, 'acc': 83.5, 'f1': 64.1}","{'auc': 0.888, 'acc': 83.7, 'f1': 62.4}","{'auc': 0.907, 'acc': 86.0, 'f1': 67.8}"
GermEval2019,"{'auc': 0.805, 'acc': 73.8, 'f1': 70.1}","{'auc': 0.894, 'acc': 82.7, 'f1': 75.1}","{'auc': 0.891, 'acc': 79.6, 'f1': 74.2}","{'auc': 0.9, 'acc': 81.6, 'f1': 74.9}","{'auc': 0.878, 'acc': 80.8, 'f1': 72.9}","{'auc': 0.904, 'acc': 83.7, 'f1': 76.6}"
GermEval2021,"{'auc': 0.703, 'acc': 62.8, 'f1': 57.3}","{'auc': 0.716, 'acc': 58.0, 'f1': 57.1}","{'auc': 0.727, 'acc': 63.1, 'f1': 58.5}","{'auc': 0.706, 'acc': 60.8, 'f1': 56.1}","{'auc': 0.708, 'acc': 57.8, 'f1': 55.7}","{'auc': 0.714, 'acc': 60.5, 'f1': 56.3}"
HateSpeechX,"{'auc': 0.593, 'acc': 67.7, 'f1': 78.8}","{'auc': 0.835, 'acc': 75.3, 'f1': 81.6}","{'auc': 0.847, 'acc': 77.8, 'f1': 82.5}","{'auc': 0.853, 'acc': 78.2, 'f1': 83.3}","{'auc': 0.837, 'acc': 76.6, 'f1': 82.7}","{'auc': 0.859, 'acc': 78.1, 'f1': 83.8}"
Sexism,"{'auc': 0.817, 'acc': 76.4, 'f1': 50.4}","{'auc': 0.878, 'acc': 82.6, 'f1': 56.1}","{'auc': 0.827, 'acc': 77.3, 'f1': 47.8}","{'auc': 0.844, 'acc': 78.1, 'f1': 50.6}","{'auc': 0.79, 'acc': 72.3, 'f1': 42.2}","{'auc': 0.894, 'acc': 80.1, 'f1': 57.4}"
US_election,"{'auc': 0.753, 'acc': 54.5, 'f1': 32.8}","{'auc': 0.807, 'acc': 83.7, 'f1': 46.5}","{'auc': 0.852, 'acc': 85.4, 'f1': 44.6}","{'auc': 0.866, 'acc': 82.1, 'f1': 47.2}","{'auc': 0.823, 'acc': 82.8, 'f1': 39.7}","{'auc': 0.847, 'acc': 84.7, 'f1': 48.0}"
ViHSD,"{'auc': 0.839, 'acc': 79.1, 'f1': 57.8}","{'auc': 0.892, 'acc': 87.4, 'f1': 64.4}","{'auc': 0.891, 'acc': 89.1, 'f1': 64.3}","{'auc': 0.883, 'acc': 86.8, 'f1': 64.5}","{'auc': 0.868, 'acc': 83.8, 'f1': 61.1}","{'auc': 0.89, 'acc': 86.3, 'f1': 65.1}"
Mean,"{'auc': 0.771, 'acc': 69.3, 'f1': 63.0}","{'auc': 0.872, 'acc': 79.0, 'f1': 69.0}","{'auc': 0.861, 'acc': 79.8, 'f1': 67.6}","{'auc': 0.861, 'acc': 78.1, 'f1': 67.7}","{'auc': 0.844, 'acc': 77.3, 'f1': 65.9}","{'auc': 0.879, 'acc': 81.9, 'f1': 70.9}"


In [21]:
result[2]["report"]

,Base,Human,Lgb,Mean,Vote,Human-Lgb
SetFit,"{'auc': 0.841, 'acc': 82.8, 'f1': 85.1}","{'auc': 0.959, 'acc': 90.5, 'f1': 90.5}","{'auc': 0.96, 'acc': 90.8, 'f1': 90.6}","{'auc': 0.968, 'acc': 91.3, 'f1': 91.0}","{'auc': 0.952, 'acc': 88.0, 'f1': 88.4}","{'auc': 0.968, 'acc': 91.4, 'f1': 91.5}"
Manueltonneau_German,"{'auc': 0.698, 'acc': 69.2, 'f1': 76.0}","{'auc': 0.879, 'acc': 79.6, 'f1': 80.3}","{'auc': 0.853, 'acc': 76.8, 'f1': 77.9}","{'auc': 0.841, 'acc': 73.8, 'f1': 77.6}","{'auc': 0.823, 'acc': 73.2, 'f1': 76.1}","{'auc': 0.865, 'acc': 77.3, 'f1': 79.5}"
HateOff,"{'auc': 0.851, 'acc': 83.5, 'f1': 85.6}","{'auc': 0.958, 'acc': 89.4, 'f1': 89.9}","{'auc': 0.96, 'acc': 91.0, 'f1': 90.9}","{'auc': 0.966, 'acc': 90.4, 'f1': 90.5}","{'auc': 0.953, 'acc': 89.3, 'f1': 89.6}","{'auc': 0.961, 'acc': 90.5, 'f1': 90.4}"
Ethos,"{'auc': 0.773, 'acc': 74.0, 'f1': 76.6}","{'auc': 0.916, 'acc': 84.2, 'f1': 82.9}","{'auc': 0.92, 'acc': 85.2, 'f1': 83.7}","{'auc': 0.917, 'acc': 83.8, 'f1': 82.5}","{'auc': 0.903, 'acc': 82.2, 'f1': 81.4}","{'auc': 0.918, 'acc': 83.6, 'f1': 82.4}"
Hasoc,"{'auc': 0.676, 'acc': 66.7, 'f1': 74.7}","{'auc': 0.837, 'acc': 73.2, 'f1': 76.9}","{'auc': 0.819, 'acc': 72.3, 'f1': 76.5}","{'auc': 0.837, 'acc': 73.6, 'f1': 77.6}","{'auc': 0.825, 'acc': 71.7, 'f1': 76.4}","{'auc': 0.827, 'acc': 74.0, 'f1': 77.2}"
GermanEval18,"{'auc': 0.551, 'acc': 48.9, 'f1': 62.4}","{'auc': 0.778, 'acc': 71.0, 'f1': 69.4}","{'auc': 0.742, 'acc': 64.8, 'f1': 66.7}","{'auc': 0.764, 'acc': 67.8, 'f1': 67.4}","{'auc': 0.756, 'acc': 65.2, 'f1': 67.4}","{'auc': 0.786, 'acc': 70.1, 'f1': 70.4}"
Mean,"{'auc': 0.735, 'acc': 71.4, 'f1': 76.6}","{'auc': 0.897, 'acc': 80.7, 'f1': 81.2}","{'auc': 0.886, 'acc': 79.7, 'f1': 80.4}","{'auc': 0.892, 'acc': 79.7, 'f1': 80.0}","{'auc': 0.874, 'acc': 77.8, 'f1': 79.3}","{'auc': 0.896, 'acc': 81.4, 'f1': 81.0}"


In [73]:
for key, value in result[1]["probs"].items():

    prob_1 = np.array(value['probs_label_1'], dtype=float)
    prob_1 = np.round(prob_1, 5)
    df_eval_set_1[model_group + "_"+ key + "_probs_label_1"] = prob_1

    prob_2 = np.array(value['probs_label_2'], dtype=float)
    prob_2 = np.round(prob_2, 5)
    df_eval_set_1[model_group + "_"+ key + "_probs_label_2"] = prob_2

for key, value in result[2]["probs"].items():

    prob_1 = np.array(value['probs_label_1'], dtype=float)
    prob_1 = np.round(prob_1, 5)
    df_eval_set_2[model_group + "_"+ key + "_probs_label_1"] = prob_1

    prob_2 = np.array(value['probs_label_2'], dtype=float)
    prob_2 = np.round(prob_2, 5)
    df_eval_set_2[model_group + "_"+ key + "_probs_label_2"] = prob_2

In [77]:
df_eval_set_1.to_csv("df_eval_set_1.csv", index=False)
df_eval_set_2.to_csv("df_eval_set_2.csv", index=False)

In [ ]:
report_all["Qwen2.5-14B_test_1"] = result[1]["report"]
report_all["Qwen2.5-14B_test_2"] = result[2]["report"]

In [70]:
report_all['Qwen2.5-14B_test_1']

,Base,Human,Lgb,Mean,Vote,Human-Lgb
Covid,"{'auc': 0.64, 'acc': 41.0, 'f1': 39.6}","{'auc': 0.865, 'acc': 86.5, 'f1': 61.3}","{'auc': 0.887, 'acc': 85.0, 'f1': 64.2}","{'auc': 0.89, 'acc': 83.5, 'f1': 64.1}","{'auc': 0.888, 'acc': 83.7, 'f1': 62.4}","{'auc': 0.907, 'acc': 86.0, 'f1': 67.8}"
GermEval2019,"{'auc': 0.805, 'acc': 73.8, 'f1': 70.1}","{'auc': 0.894, 'acc': 82.7, 'f1': 75.1}","{'auc': 0.891, 'acc': 79.6, 'f1': 74.2}","{'auc': 0.9, 'acc': 81.6, 'f1': 74.9}","{'auc': 0.878, 'acc': 80.8, 'f1': 72.9}","{'auc': 0.904, 'acc': 83.7, 'f1': 76.6}"
GermEval2021,"{'auc': 0.703, 'acc': 62.8, 'f1': 57.3}","{'auc': 0.716, 'acc': 58.0, 'f1': 57.1}","{'auc': 0.727, 'acc': 63.1, 'f1': 58.5}","{'auc': 0.706, 'acc': 60.8, 'f1': 56.1}","{'auc': 0.708, 'acc': 57.8, 'f1': 55.7}","{'auc': 0.714, 'acc': 60.5, 'f1': 56.3}"
HateSpeechX,"{'auc': 0.593, 'acc': 67.7, 'f1': 78.8}","{'auc': 0.835, 'acc': 75.3, 'f1': 81.6}","{'auc': 0.847, 'acc': 77.8, 'f1': 82.5}","{'auc': 0.853, 'acc': 78.2, 'f1': 83.3}","{'auc': 0.837, 'acc': 76.6, 'f1': 82.7}","{'auc': 0.859, 'acc': 78.1, 'f1': 83.8}"
Sexism,"{'auc': 0.817, 'acc': 76.4, 'f1': 50.4}","{'auc': 0.878, 'acc': 82.6, 'f1': 56.1}","{'auc': 0.827, 'acc': 77.3, 'f1': 47.8}","{'auc': 0.844, 'acc': 78.1, 'f1': 50.6}","{'auc': 0.79, 'acc': 72.3, 'f1': 42.2}","{'auc': 0.894, 'acc': 80.1, 'f1': 57.4}"
US_election,"{'auc': 0.753, 'acc': 54.5, 'f1': 32.8}","{'auc': 0.807, 'acc': 83.7, 'f1': 46.5}","{'auc': 0.852, 'acc': 85.4, 'f1': 44.6}","{'auc': 0.866, 'acc': 82.1, 'f1': 47.2}","{'auc': 0.823, 'acc': 82.8, 'f1': 39.7}","{'auc': 0.847, 'acc': 84.7, 'f1': 48.0}"
ViHSD,"{'auc': 0.839, 'acc': 79.1, 'f1': 57.8}","{'auc': 0.892, 'acc': 87.4, 'f1': 64.4}","{'auc': 0.891, 'acc': 89.1, 'f1': 64.3}","{'auc': 0.883, 'acc': 86.8, 'f1': 64.5}","{'auc': 0.868, 'acc': 83.8, 'f1': 61.1}","{'auc': 0.89, 'acc': 86.3, 'f1': 65.1}"
Mean,"{'auc': 0.771, 'acc': 69.3, 'f1': 63.0}","{'auc': 0.872, 'acc': 79.0, 'f1': 69.0}","{'auc': 0.861, 'acc': 79.8, 'f1': 67.6}","{'auc': 0.861, 'acc': 78.1, 'f1': 67.7}","{'auc': 0.844, 'acc': 77.3, 'f1': 65.9}","{'auc': 0.879, 'acc': 81.9, 'f1': 70.9}"


In [69]:
report_all['Qwen2.5-14B_test_2']

,Base,Human,Lgb,Mean,Vote,Human-Lgb
SetFit,"{'auc': 0.841, 'acc': 82.8, 'f1': 85.1}","{'auc': 0.959, 'acc': 90.5, 'f1': 90.5}","{'auc': 0.96, 'acc': 90.8, 'f1': 90.6}","{'auc': 0.968, 'acc': 91.3, 'f1': 91.0}","{'auc': 0.952, 'acc': 88.0, 'f1': 88.4}","{'auc': 0.968, 'acc': 91.4, 'f1': 91.5}"
Manueltonneau_German,"{'auc': 0.698, 'acc': 69.2, 'f1': 76.0}","{'auc': 0.879, 'acc': 79.6, 'f1': 80.3}","{'auc': 0.853, 'acc': 76.8, 'f1': 77.9}","{'auc': 0.841, 'acc': 73.8, 'f1': 77.6}","{'auc': 0.823, 'acc': 73.2, 'f1': 76.1}","{'auc': 0.865, 'acc': 77.3, 'f1': 79.5}"
HateOff,"{'auc': 0.851, 'acc': 83.5, 'f1': 85.6}","{'auc': 0.958, 'acc': 89.4, 'f1': 89.9}","{'auc': 0.96, 'acc': 91.0, 'f1': 90.9}","{'auc': 0.966, 'acc': 90.4, 'f1': 90.5}","{'auc': 0.953, 'acc': 89.3, 'f1': 89.6}","{'auc': 0.961, 'acc': 90.5, 'f1': 90.4}"
Ethos,"{'auc': 0.773, 'acc': 74.0, 'f1': 76.6}","{'auc': 0.916, 'acc': 84.2, 'f1': 82.9}","{'auc': 0.92, 'acc': 85.2, 'f1': 83.7}","{'auc': 0.917, 'acc': 83.8, 'f1': 82.5}","{'auc': 0.903, 'acc': 82.2, 'f1': 81.4}","{'auc': 0.918, 'acc': 83.6, 'f1': 82.4}"
Hasoc,"{'auc': 0.676, 'acc': 66.7, 'f1': 74.7}","{'auc': 0.837, 'acc': 73.2, 'f1': 76.9}","{'auc': 0.819, 'acc': 72.3, 'f1': 76.5}","{'auc': 0.837, 'acc': 73.6, 'f1': 77.6}","{'auc': 0.825, 'acc': 71.7, 'f1': 76.4}","{'auc': 0.827, 'acc': 74.0, 'f1': 77.2}"
GermanEval18,"{'auc': 0.551, 'acc': 48.9, 'f1': 62.4}","{'auc': 0.778, 'acc': 71.0, 'f1': 69.4}","{'auc': 0.742, 'acc': 64.8, 'f1': 66.7}","{'auc': 0.764, 'acc': 67.8, 'f1': 67.4}","{'auc': 0.756, 'acc': 65.2, 'f1': 67.4}","{'auc': 0.786, 'acc': 70.1, 'f1': 70.4}"
Mean,"{'auc': 0.735, 'acc': 71.4, 'f1': 76.6}","{'auc': 0.897, 'acc': 80.7, 'f1': 81.2}","{'auc': 0.886, 'acc': 79.7, 'f1': 80.4}","{'auc': 0.892, 'acc': 79.7, 'f1': 80.0}","{'auc': 0.874, 'acc': 77.8, 'f1': 79.3}","{'auc': 0.896, 'acc': 81.4, 'f1': 81.0}"
